# Датасет содержит следующие признаки:

    age — возраст
    sex — пол (1 - мужчина, 0 - женщина)
    cp — тип боли в груди (4 значения)
    trestbps — артериальное давление в покое
    chol — холестерин сыворотки в мг/дл
    fbs — уровень сахара в крови натощак > 120 мг/дл
    restecg — результаты электрокардиографии в покое (значения 0,1,2)
    thalach — достигнута максимальная частота сердечных сокращений
    exang — стенокардия, вызванная физической нагрузкой
    oldpeak — депрессия ST, вызванная физической нагрузкой, по сравнению с состоянием покоя
    slope — наклон пикового сегмента ST при нагрузке
    ca — количество крупных сосудов (0-3), окрашенных при флюроскопии
    thal — дефект, где 3 = нормальный; 6 = фиксированный дефект; 7 = обратимый дефект


In [1]:
import pandas as pd

heart = pd.read_csv('data/heart.csv')

In [2]:
heart.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [4]:
heart['old'] = heart['age'] > 60
heart['old'].sum()

79

In [6]:
def systolic_norm(age: float, sex: int) -> int:
    # sex: 1 — мужчина, 0 — женщина
    bands = [
        (20, {1: 123, 0: 116}),
        (30, {1: 126, 0: 120}),
        (40, {1: 129, 0: 127}),
        (50, {1: 135, 0: 137}),
        (60, {1: 142, 0: 144}),
        (1e9, {1: 142, 0: 159}),  # 61+
    ]
    for upper, values in bands:
        if age <= upper:
            return values[sex]
    raise ValueError("Unexpected age")

heart["trestbps_mean"] = heart.apply(
    lambda row: systolic_norm(row["age"], int(row["sex"])), axis=1
)

print(heart.loc[300, ["age", "sex", "trestbps_mean"]])

age               68
sex                1
trestbps_mean    142
Name: 300, dtype: object


In [8]:
categorical = ["cp", "restecg", "slope", "ca", "thal"]
heart = pd.get_dummies(heart, columns=categorical, dtype=int)

print(f"Всего признаков: {heart.shape[1]}")

Всего признаков: 30


In [9]:
heart.head()

,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,target,old,...,slope_2,ca_0,ca_1,ca_2,ca_3,ca_4,thal_0,thal_1,thal_2,thal_3
0,63,1,145,233,1,150,0,2.3,1,True,...,0,1,0,0,0,0,0,1,0,0
1,37,1,130,250,0,187,0,3.5,1,False,...,0,1,0,0,0,0,0,0,1,0
2,41,0,130,204,0,172,0,1.4,1,False,...,1,1,0,0,0,0,0,0,1,0
3,56,1,120,236,0,178,0,0.8,1,False,...,1,1,0,0,0,0,0,0,1,0
4,57,0,120,354,0,163,1,0.6,1,False,...,1,1,0,0,0,0,0,0,1,0


In [12]:
# для нормализации, стандартизации
from sklearn import preprocessing

# инициализируем нормализатор RobustScaler
r_scaler = preprocessing.RobustScaler()
col_names = list(heart.columns)

# копируем исходный датасет
heart_r = r_scaler.fit_transform(heart)

heart_r = pd.DataFrame(heart_r, columns=col_names)

# смотрим описательные статистики, ответ 0.816232
heart_r.describe().loc['std', 'chol']

0.8162322990225203